In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import gymnasium as gym
import time


In [ ]:
print(torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


In [ ]:
class ActorCritic(nn.Module):
	def __init__(self, state_dim: int, action_dim):
		super().__init__()
		
		# Aprende información común del entorno
		self.shared = nn.Sequential(
			nn.Linear(state_dim, 128),
			nn.ReLU()
		)
		
		# Devuelve las probabilidades de cada acción (izquierda, derecha)
		self.actor = nn.Sequential(
			nn.Linear(128, action_dim),
			nn.Softmax(dim=-1)
		)

		# Crítico (devuelve el valor V)
		# V -> valor de retorno que obtiene un agente empezando en un estado s concreto y siguiendo una política específica
		self.critic = nn.Linear(128, 1)

	def forward(self, state):
		x = self.shared(state)
				
		policy = self.actor(x)
		
		value = self.critic(x)

		return policy, value


In [ ]:
env = gym.make("CartPole-v1")

state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

print(state_dim, action_dim)


In [41]:
model = ActorCritic(state_dim, action_dim).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)


In [42]:
gamma = 0.99
num_episodes = 500

recent_rewards = []

for episode in range(num_episodes):

	state, _ = env.reset()

	done = False
	episode_reward = 0

	states = []
	actions = []
	rewards = []
	log_probs = []
	values = []
	entropies = []

	while not done:
		state_tensor = torch.tensor(
			state,
			dtype=torch.float32,
			device=device
		)
		
		policy, value = model(state_tensor)
		dist = Categorical(policy)

		action = dist.sample()

		log_prob = dist.log_prob(action)
		entropy = dist.entropy()
		
		next_state, reward, terminated, truncated, _ = env.step(action.item())
		done = terminated or truncated

		states.append(state_tensor)
		actions.append(action)
		rewards.append(reward)
		log_probs.append(log_prob)
		values.append(value)
		entropies.append(entropy)

		episode_reward += reward
		state = next_state

	returns = []
	G = 0

	# Montecarlo -> se calcula el retorno (G) para el episodio completo
	for r in reversed(rewards):
		G = r + gamma * G
		returns.insert(0, G)
	
	returns = torch.tensor(returns, device=device)
	values = torch.stack(values).squeeze()

	advantages = returns - values
	advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

	log_probs = torch.stack(log_probs)
	entropies = torch.stack(entropies)

	actor_loss = -(log_probs * advantages.detach()).mean()
	critic_loss = advantages.pow(2).mean()
	entropy_loss = entropies.mean()

	loss = actor_loss + 0.5 * critic_loss - 0.001 * entropy_loss
		
	optimizer.zero_grad()
	loss.backward()
	optimizer.step()
		
	recent_rewards.append(episode_reward)

	if len(recent_rewards) > 100:
		recent_rewards.pop(0)

	avg_reward = sum(recent_rewards) / len(recent_rewards)

	print(
		f"Episode {episode+1:4d}/{num_episodes} | "
		f"Reward: {episode_reward:6.1f} | "
		f"Avg100: {avg_reward:7.2f} | "
		f"Actor: {actor_loss.item():8.4f} | "
		f"Critic: {critic_loss.item():8.4f} | "
		f"Entropy: {entropy_loss.item():8.4f}"
	)

env.close()


Episode    1/500 | Reward:   14.0 | Avg100:   14.00 | Actor:   0.0051 | Critic:   0.9286 | Entropy:   0.6824
Episode    2/500 | Reward:   13.0 | Avg100:   13.50 | Actor:   0.0590 | Critic:   0.9231 | Entropy:   0.6452
Episode    3/500 | Reward:   19.0 | Avg100:   15.33 | Actor:  -0.0053 | Critic:   0.9474 | Entropy:   0.6895
Episode    4/500 | Reward:   19.0 | Avg100:   16.25 | Actor:   0.0011 | Critic:   0.9474 | Entropy:   0.6879
Episode    5/500 | Reward:   27.0 | Avg100:   18.40 | Actor:   0.0142 | Critic:   0.9630 | Entropy:   0.6837
Episode    6/500 | Reward:   17.0 | Avg100:   18.17 | Actor:   0.0744 | Critic:   0.9412 | Entropy:   0.6653
Episode    7/500 | Reward:   26.0 | Avg100:   19.29 | Actor:   0.0216 | Critic:   0.9615 | Entropy:   0.6839
Episode    8/500 | Reward:   25.0 | Avg100:   20.00 | Actor:   0.0073 | Critic:   0.9600 | Entropy:   0.6745
Episode    9/500 | Reward:   15.0 | Avg100:   19.44 | Actor:  -0.0029 | Critic:   0.9333 | Entropy:   0.6892
Episode   10/500 | 

In [ ]:
gamma = 0.99
num_episodes = 2000

recent_rewards = []

for episode in range(num_episodes):

    state, _ = env.reset()

    done = False
    episode_reward = 0

    actor_losses = []
    critic_losses = []
    entropies = []

    while not done:

        state_tensor = torch.tensor(
            state,
            dtype=torch.float32,
            device=device
        )

        policy, value = model(state_tensor)

        value = value.squeeze()

        dist = Categorical(policy)
        action = dist.sample()

        log_prob = dist.log_prob(action)
        entropy = dist.entropy()

        next_state, reward, terminated, truncated, _ = env.step(action.item())

        done = terminated or truncated

        next_state_tensor = torch.tensor(
            next_state,
            dtype=torch.float32,
            device=device
        )

        with torch.no_grad():
            _, next_value = model(next_state_tensor)
            next_value = next_value.squeeze()

        # Diferencias temporales -> se aprende en cada paso un poquito
        td_target = reward + gamma * next_value * (1 - int(done))

        advantage = td_target - value

        actor_loss = -(log_prob * advantage.detach())

        critic_loss = advantage.pow(2)

        actor_losses.append(actor_loss)
        critic_losses.append(critic_loss)
        entropies.append(entropy)

        episode_reward += reward
        state = next_state

    actor_loss = torch.stack(actor_losses).mean()
    critic_loss = torch.stack(critic_losses).mean()
    entropy_loss = torch.stack(entropies).mean()

    loss = actor_loss + 0.5 * critic_loss - 0.001 * entropy_loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    recent_rewards.append(episode_reward)

    if len(recent_rewards) > 100:
        recent_rewards.pop(0)

    avg_reward = sum(recent_rewards) / len(recent_rewards)

    print(
        f"Episode {episode+1:4d}/{num_episodes} | "
        f"Reward: {episode_reward:6.1f} | "
        f"Avg100: {avg_reward:7.2f} | "
        f"Actor: {actor_loss.item():8.4f} | "
        f"Critic: {critic_loss.item():8.4f} | "
        f"Entropy: {entropy_loss.item():8.4f}"
    )

env.close()


In [45]:
env = gym.make("CartPole-v1", render_mode="human")

state, _ = env.reset()
done = False

while not done:
	state_tensor = torch.tensor(
		state,
		dtype=torch.float32,
		device=device
	)

	with torch.no_grad():
		policy, _ = model(state_tensor)

	action = torch.argmax(policy).item()

	state, reward, terminated, truncated, _ = env.step(action)

	done = terminated or truncated

	time.sleep(0.02)

env.close()


In [ ]:
# CartPole
state_dim = 4
action_dim = 2

model = ActorCritic(state_dim, action_dim)

optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Valor de retorno
# 0 -> prioriza recompensas inmediatas
# 1 -> prioriza recompensas lejanas
gamma = 0.99


In [ ]:
state = torch.FloatTensor([0.1, 0.2, 0.3, 0.4])
next_state = torch.FloatTensor([0.15, 0.25, 0.35, 0.45])

reward = torch.tensor(1.0)
done = torch.tensor(0.0)

policy, value = model(state)

print(policy)

dist = Categorical(policy)

action = dist.sample()

print(action)

log_prob = dist.log_prob(action)

print(log_prob)

_, next_value = model(next_state)

# Done = 1 -> finaliza el episodio, 0 -> continúa jugando
# Se obtiene el valor Q(s, a) aproximando por Montecarlo
td_target = reward + gamma * next_value * (1 - done)

# advantage -> cómo de buena es esa acción respecto en este estado.
# Para ello se resta el valor Q(s, a) - V(s)
advantage = td_target - value

# log_prob -> cuánto cambios realiza esa acción (es cómo el descenso de gradiente)
actor_loss = -log_prob * advantage.detach()

# El crítico simplemente quiere evaluar el mejor estado posible
critic_loss = advantage.pow(2)

loss = actor_loss + critic_loss

optimizer.zero_grad()
loss.backward()
optimizer.step()

print("Action:", action.item())
print("Value:", value.item())
print("Advantage:", advantage.item())
